# GTEx model building with PLIER

💡 **Environment:** `clamp-analyses`  

This notebook builds latent variable models from GTEx v8 RNA‑seq TPM data using PLIER. It automates downloading and preprocessing the GTEx matrix, creates a Filebacked Big Matrix (FBM), computes an SVD to estimate the model dimension, prepares pathway priors, runs PLIER, and saves model outputs (B, Z, summaries) and intermediate files. Configuration and paths are controlled via `config.R`.

## Load libraries

In [1]:
# Create a timestamp to track the start of the analysis
start_time <- Sys.time()
cat("GTEx CLAMP and PLIER analysis started at:", format(start_time), "\n")

GTEx CLAMP and PLIER analysis started at: 2026-01-24 13:14:43 


In [2]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))

set.seed(config$GTEx$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loadin

## Output directory

In [3]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

output_data_dir

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex"

## Input

In [4]:
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))
gtex_svdRes <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))
CLAMP_K_gtex <- readRDS(file.path(output_data_dir, "CLAMP_K_gtex.rds"))
gtex_genes <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))
samples <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))

# Settings

In [5]:
block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$GTEx$N_CORES

## Prepare pathway priors

In [6]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

# prefix each gene‐set name with its library to guarantee uniqueness
for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways

Inverting...

done



# PLIER

Run PLIER with the same inputs

In [7]:
gtex_plier = PLIER::PLIER(
    gtex_fbm_filt[], 
    as.matrix(gtex_matched), 
    svdres = gtex_svdRes, 
    Chat = as.matrix(gtex_chatObj), 
    doCrossval = TRUE, 
    k = CLAMP_K_gtex
  )

Removing 0 pathways with too few genes



[1] 135.8334
[1] "L2 is set to 135.833443715207"
[1] "L1 is set to 67.9167218576034"


errorY (SVD based:best possible) = 0.3614

New L3 is 0.000158461325115751

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000108908769855066

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.00012340980408668

New L3 is 0.000139841628594101

New L3 is 0.00012340980408668

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

Bdiff is not decreasing

converged at  iteration 185 Bdiff is not decreasing

There are 155  LVs with AUC>0.70



In [8]:
colnames(gtex_plier$Z) <- paste0('LV', seq_len(ncol(gtex_plier$Z)))

In [9]:
head(gtex_plier$Z)
dim(gtex_plier$Z)

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV403,LV404,LV405,LV406,LV407,LV408,LV409,LV410,LV411,LV412
WASH7P,0.00000000,0.21239814,0.013920330,0.00000000,0.10776016,0.04514973,0.04666207,0.00000000,0,0.000000000,⋯,0.15322629,0.00000000,0.11773388,0.02124340,0,0.00000000,0.02090412,0.00000000,0.00000000,0.0000000000
RP11-34P13.15,0.00000000,0.08016658,0.000000000,0.73023609,0.02143773,0.32710014,0.00000000,0.07640915,0,0.000000000,⋯,0.14737412,0.15361652,0.01766793,0.06805750,0,0.00000000,0.37455197,0.00000000,0.02969857,0.0000000000
RP11-34P13.16,0.07959466,0.00190165,0.013202172,0.77884850,0.01387984,0.31223373,0.00000000,0.09853801,0,0.000000000,⋯,0.09371039,0.23224688,0.01675043,0.06187363,0,0.00000000,0.40869678,0.00000000,0.03896699,0.0000000000
RP11-34P13.18,0.07872883,0.00000000,0.000000000,0.00000000,0.07121090,0.18285201,0.00000000,0.00000000,0,0.000000000,⋯,0.00000000,0.01790337,0.00000000,0.00000000,0,0.05127827,0.00000000,0.00000000,0.00000000,0.0006403351
AP006222.2,0.00000000,0.00000000,0.007406785,0.03941823,0.05031049,0.07873692,0.00000000,0.01805285,0,0.000000000,⋯,0.00000000,0.16397922,0.00000000,0.07181693,0,0.00000000,0.36522995,0.07277403,0.00000000,0.1402921060
MTND1P23,0.09431164,0.00000000,0.025999211,0.07196043,0.10902242,0.01829400,0.00000000,0.00000000,0,0.009574933,⋯,0.00000000,0.00000000,0.00000000,0.16764503,0,0.04277822,0.00000000,0.01404362,0.00000000,0.0000000000


[1] 21613   412

In [10]:
gtex_plier$summary <- gtex_plier$summary %>%
    dplyr::rename(LV = `LV index`)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

In [11]:
head(gtex_plier$summary)

,pathway,LV,AUC,p-value,FDR
,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,BP_ATP Metabolic Process (GO:0046034),LV1,0.6488955,5.906709e-02,1.232369e-01
2,BP_COPII-coated Vesicle Budding (GO:0090114),LV1,0.8445930,7.121746e-04,4.947113e-03
3,BP_DNA Integrity Checkpoint Signaling (GO:0031570),LV1,0.8186892,8.067132e-04,5.361739e-03
4,BP_DNA Recombination (GO:0006310),LV1,0.9129958,2.188187e-05,3.341812e-04
5,BP_DNA Repair (GO:0006281),LV1,0.8190727,3.710146e-17,1.185534e-14
6,BP_DNA-templated Transcription (GO:0006351),LV1,0.6925706,2.118287e-05,3.246999e-04


In [12]:
head(gtex_plier$B)
dim(gtex_plier$B)

"1,BP_Import Into Nucleus (GO:0051170)",0.170241356,-0.09351935,0.23693459,-0.01180202,-0.281061217,0.04636526,0.11271615,0.18181215,0.105277359,0.215906802,⋯,0.039258141,0.27624064,0.226091179,0.03840126,0.40333519,0.42289422,0.51833836,0.35319105,0.209505486,0.14079887
LV 2,0.045196082,0.03320152,0.10252546,0.03884325,-0.012507648,0.01493169,-0.05944762,0.02367400,0.007804151,0.019760294,⋯,-0.009809677,-0.02550467,0.006608416,-0.03739649,-0.03545184,0.01610731,-0.01555673,0.01608860,-0.005309853,-0.03742639
"3,BP_Monocarboxylic Acid Biosynthetic Process (GO:0072330)",-0.140843264,0.15121471,0.17974016,-0.10928469,-0.013626995,0.08483139,0.07269954,-0.05447547,-0.125698759,-0.000148313,⋯,-0.059706055,-0.10224740,-0.198656048,-0.12754846,0.04007009,0.03541707,0.10998136,0.04643466,0.063063376,-0.06852731
"4,BP_Neutrophil Chemotaxis (GO:0030593)",-0.129072123,0.03872968,-0.06930013,-0.19374085,0.010492270,-0.21361938,-0.06764105,-0.16963700,-0.102800670,-0.141545927,⋯,0.019268916,-0.16436923,-0.091509593,-0.05166002,-0.08406586,-0.09493653,-0.08792751,0.02317060,0.035171083,-0.00849022
"5,BP_Skeletal Muscle Tissue Development (GO:0007519)",-0.008728672,1.11572549,0.06141888,-0.03627121,0.002387445,-0.10423248,-0.04207220,-0.11128401,0.005730502,-0.012504622,⋯,0.103494414,-0.13437529,-0.116120214,-0.10675388,0.12750393,-0.26743699,-0.12856121,0.09412401,2.122929541,0.10336626
"6,BP_Cilium Assembly (GO:0060271)",-0.034616854,-0.03722485,-0.08974681,-0.10947183,-0.088209031,-0.11615536,-0.08753346,-0.11457714,-0.071312143,-0.137374961,⋯,-0.082789851,-0.13723425,-0.044550325,-0.07546499,-0.05170813,-0.19238297,-0.12969105,-0.10765660,-0.067002883,-0.08360771


[1]   412 17382

In [13]:
colnames(gtex_plier$B) <- samples

In [14]:
saveRDS(gtex_plier, file = file.path(output_data_dir, "PLIER_BP.rds"))

In [15]:
model_dir <- file.path(output_data_dir, "PLIER_BP")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_plier$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- gtex_plier$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- gtex_plier$summary
write.csv(summary, file.path(model_dir, "summary.csv"))